In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from scipy import stats
from src.data_loader import DataLoader

# Load cleaned data
loader = DataLoader(data_dir="../data")
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv")

# Re-apply categorical dtype for the clinical flag/category columns
# (dtype doesn't survive a plain CSV round-trip, so we restore it here
# before running numeric-only summary/normality/correlation analysis)
for col in df.columns:
    if df[col].nunique() < 10:
        df[col] = df[col].astype('category')

In [2]:
# 1. Comprehensive Summary Statistics
def comprehensive_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for all numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    stats_dict = {
        'Column': [], 'Mean': [], 'Median': [], 'Mode': [], 'Std': [],
        'Variance': [], 'Skewness': [], 'Kurtosis': [], 'Range': [],
        'IQR': [], 'Q1': [], 'Q3': [], 'CV': [], 'Missing %': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        stats_dict['Column'].append(col)
        stats_dict['Mean'].append(data.mean())
        stats_dict['Median'].append(data.median())
        stats_dict['Mode'].append(data.mode()[0] if not data.mode().empty else np.nan)
        stats_dict['Std'].append(data.std())
        stats_dict['Variance'].append(data.var())
        stats_dict['Skewness'].append(data.skew())
        stats_dict['Kurtosis'].append(data.kurtosis())
        stats_dict['Range'].append(data.max() - data.min())
        stats_dict['IQR'].append(data.quantile(0.75) - data.quantile(0.25))
        stats_dict['Q1'].append(data.quantile(0.25))
        stats_dict['Q3'].append(data.quantile(0.75))
        stats_dict['CV'].append(data.std() / data.mean() if data.mean() != 0 else np.nan)
        stats_dict['Missing %'].append(df[col].isnull().mean() * 100)
    
    return pd.DataFrame(stats_dict)

# Generate stats
stats_df = comprehensive_stats(df)
stats_df.to_csv('../reports/summary_statistics.csv', index=False)
print("Summary Statistics:")
print(stats_df.to_string())

Summary Statistics:
           Column        Mean      Median   Mode        Std     Variance  Skewness  Kurtosis       Range        IQR          Q1          Q3        CV  Missing %
0             age   54.420530   55.500000   58.0   9.047970    81.865757 -0.203743 -0.527512   48.000000  13.000000   48.000000   61.000000  0.166260        0.0
1        trestbps  131.258278  130.000000  120.0  16.605232   275.733735  0.389876 -0.158677   76.000000  20.000000  120.000000  140.000000  0.126508        0.0
2            chol  245.377070  240.500000  197.0  47.486683  2254.985097  0.335700 -0.091647  244.375000  63.750000  211.000000  274.750000  0.193525        0.0
3         thalach  149.612997  152.500000  162.0  22.765983   518.289982 -0.490122 -0.235395  117.875000  32.750000  133.250000  166.000000  0.152166        0.0
4         oldpeak    1.027815    0.800000    0.0   1.110395     1.232978  0.993897  0.115932    4.000000   1.600000    0.000000    1.600000  1.080346        0.0
5      hr_rese

In [3]:
# 2. Distribution Tests
def test_normality(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Perform Shapiro-Wilk test for normality on numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    results = {
        'Column': [], 'Statistic': [], 'P-value': [], 'Normal?': [], 'Interpretation': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 3:
            stat, p_value = stats.shapiro(data)
            is_normal = p_value > alpha
            results['Column'].append(col)
            results['Statistic'].append(stat)
            results['P-value'].append(p_value)
            results['Normal?'].append(is_normal)
            results['Interpretation'].append("Normal" if is_normal else "Not normal")
    
    return pd.DataFrame(results)

normality_df = test_normality(df)
normality_df.to_csv('../reports/normality_tests.csv', index=False)
print("\nNormality Tests:")
print(normality_df.to_string())


Normality Tests:
           Column  Statistic       P-value  Normal? Interpretation
0             age   0.986637  6.744821e-03    False     Not normal
1        trestbps   0.974259  3.036506e-05    False     Not normal
2            chol   0.988688  1.879994e-02    False     Not normal
3         thalach   0.976908  8.670523e-05    False     Not normal
4         oldpeak   0.852779  2.644797e-16    False     Not normal
5      hr_reserve   0.961919  4.121429e-07    False     Not normal
6  chol_age_ratio   0.981645  6.501694e-04    False     Not normal

In [4]:
# 3. Correlation Analysis
def correlation_analysis(df: pd.DataFrame) -> dict:
    """
    Calculate correlation matrices using different methods.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    corr_matrix = df[numeric_cols].corr()
    spearman_matrix = df[numeric_cols].corr(method='spearman')
    
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_value = corr_matrix.iloc[i, j]
            corr_pairs.append({
                'Variable 1': corr_matrix.columns[i],
                'Variable 2': corr_matrix.columns[j],
                'Correlation': corr_value,
                'Strength': abs(corr_value)
            })
    
    corr_pairs = sorted(corr_pairs, key=lambda x: x['Strength'], reverse=True)
    
    return {
        'pearson': corr_matrix,
        'spearman': spearman_matrix,
        'top_correlations': corr_pairs[:10]
    }

corr_results = correlation_analysis(df)

corr_results['pearson'].to_csv('../reports/pearson_correlation.csv')
corr_results['spearman'].to_csv('../reports/spearman_correlation.csv')

print("\nTop 5 Correlations:")
for i, pair in enumerate(corr_results['top_correlations'][:5], 1):
    print(f"{i}. {pair['Variable 1']} <-> {pair['Variable 2']}: {pair['Correlation']:.3f}")


Top 5 Correlations:
1. thalach <-> hr_reserve: -0.918
2. chol <-> chol_age_ratio: 0.659
3. age <-> chol_age_ratio: -0.584
4. age <-> thalach: -0.395
5. thalach <-> oldpeak: -0.349